# GHDNN | Freeze the IQ-OTH/NCCD Model for External Validation

**Purpose:** Fit **one final frozen 3-class model** using the existing full-resolution 64-granule GoogLeNet features from the completed Kaggle experiment. **Do not train on LUNGx**. Outputs `GHDNN_FROZEN_MODEL.zip` for the later external evaluation. The feature extractor remains frozen ImageNet-pretrained GoogLeNet. Patient-level independence is not established by the IQ-OTH/NCCD data.

**Run only after the full 224×224 extraction notebook has successfully saved** `features_64_224.npy` and `labels.npy`. This is a deployment fit on all original dataset images, **not an additional independent accuracy experiment**.


In [ ]:
from pathlib import Path
import numpy as np, sklearn, joblib, json, hashlib, zipfile, time
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from collections import Counter

# Previous Kaggle notebook's cache must be visible in working or attached as an input
candidate_roots=[Path('/kaggle/working/ghdnn_googlenet_matched_224')]
for root in (Path('/kaggle/input'), Path('/kaggle/working')):
    if root.exists():
        candidate_roots += [p.parent for p in root.rglob('features_64_224.npy')]
source = next((r for r in candidate_roots if (r/'features_64_224.npy').is_file() and (r/'labels.npy').is_file()), None)
if source is None:
    raise FileNotFoundError('Previous extracted features not visible. Open the original Kaggle notebook or add its saved output as an Input; require features_64_224.npy and labels.npy. Do NOT rerun extraction unnecessarily.')
print('SOURCE:',source)
Z=np.load(source/'features_64_224.npy', mmap_mode='r')
y=np.load(source/'labels.npy')
assert Z.shape==(1054,64,1024), f'Unexpected feature shape: {Z.shape}'
assert y.shape==(1054,) and np.array_equal(np.bincount(y,minlength=3),[102,547,405]), 'Unexpected labels; stop and inspect'
meta=json.loads((source/'audit.json').read_text()) if (source/'audit.json').exists() else {}
assert meta.get('image_size') in (None,224),'Wrong image size'
assert meta.get('network_input') in (None,224),'Wrong GoogLeNet input size'
assert meta.get('preprocessing') in (None,'ImageNet normalization followed by GoogLeNet transform_input=True'), 'Preprocessing mismatch'
print('VERIFIED',Z.shape,'classes',np.bincount(y),'source metadata',meta)


In [ ]:
# Fit PCA ONLY on the original development dataset; nothing from LUNGx is used.
# This duplicates the PCA->Scaler->SVM fitting steps used in the internal 64-granule baseline.
t0=time.time()
Xflat=np.asarray(Z).reshape(-1,1024)
pca=PCA(n_components=64,svd_solver='randomized',random_state=2026)
pca.fit(Xflat)
del Xflat
reduced=np.empty((len(y),64*64),dtype='float32')
for st in range(0,len(y),48):
    en=min(st+48,len(y))
    chunk=np.asarray(Z[st:en]).reshape(-1,1024)
    reduced[st:en]=pca.transform(chunk).reshape(en-st,-1).astype('float32')
scaler=StandardScaler().fit(reduced)
X=scaler.transform(reduced)
classifier=SVC(C=1.0,gamma='scale',kernel='rbf',class_weight='balanced',decision_function_shape='ovr',probability=True,random_state=2026)
classifier.fit(X,y)
print('FIT FINISHED',len(classifier.support_), 'support vectors', 'elapsed sec',round(time.time()-t0,1))

output=Path('/kaggle/working/GHDNN_External_Frozen')
output.mkdir(parents=True,exist_ok=True)
joblib.dump({'pca':pca, 'scaler':scaler, 'svm':classifier, 'classes':['Benign','Malignant','Normal'], 'n_granules':64,'raw_feature_size':1024,'pc_dim':64,'patch_size':28,'image_size':224,'network_input':224, 'transform_input':True}, output/'ghdnn_frozen_pipeline.joblib',compress=3)
manifest={
 'source': 'IQ-OTH/NCCD, 1054 SHA256-unique CT images',
 'input_feature_shape':list(Z.shape),
 'training_count':int(len(y)),
 'counts':np.bincount(y,minlength=3).tolist(),
 'classes':['Benign','Malignant','Normal'],
 'feature_extractor':'Frozen Torchvision GoogLeNet IMAGENET1K_V1, 224x224 per patch, official transform_input=True',
 'image_resized_to':[224,224], 'granule_grid':[8,8], 'patch_original_size':[28,28],
 'spatial_pca_components':64, 'selection':'No selection: concatenate all 64 positions in fixed row-major order',
 'classifier':'StandardScaler + RBF SVM C=1 gamma=scale balanced; fitted on IQ-OTH/NCCD only',
 'limitations':['Not patient-independent', 'No external results yet','3-class full-slice model not identical to LUNGx nodule-level binary task', 'ImageNet pretraining, not CNN CT fine-tuning'],
 'feature_cache_provenance':meta,
 'python_sklearn_version':sklearn.__version__,
 'probability_note':'SVM probability calibration is learned from IQ-OTH/NCCD only; do not use LUNGx labels for threshold tuning.'
}
(output/'manifest.json').write_text(json.dumps(manifest,indent=2,ensure_ascii=False))
(output/'external_evaluation_protocol.txt').write_text("""LUNGx is a different task (benign versus malignant annotated nodules) than IQ-OTH/NCCD (3-class CT slices). This model is frozen and must not be tuned using LUNGx test labels. Acquire official LUNGx DICOM and location/diagnosis spreadsheets from TCIA/IDC. Match nodule coordinates to the correct slice and define in advance a full-slice or ROI policy. Report normal-class predictions separately (out-of-task), do not discard them. Restrict malignant-versus-benign AUROC to predeclared probability rule. Never claim this is 3-class independent validation. Include patient/scan clustering for uncertainty; multiple nodules can share a scan. See https://www.cancerimagingarchive.net/collection/spie-aapm-lung-ct-challenge/ and https://portal.imaging.datacommons.cancer.gov/collections/spie_aapm_lung_ct_challenge/
""")
zipfile_path=Path('/kaggle/working/GHDNN_FROZEN_MODEL.zip')
with zipfile.ZipFile(zipfile_path,'w',compression=zipfile.ZIP_DEFLATED) as archive:
    for f in sorted(output.iterdir()): archive.write(f,arcname=f.name)
print('READY TO DOWNLOAD:',zipfile_path, 'size MB:',round(zipfile_path.stat().st_size/1e6,2))


**Important:** The notebook above intentionally does **not** invent external validation results or retrain on LUNGx. Once the frozen model is saved, the next step is the independent LUNGx dataset (public DICOM ~12 GB, with true diagnosis and nodule coordinates). We must implement and predeclare how to map its binary annotated-nodule task to this three-class CT-image model before evaluating.
